# Huni ground truth, timed against the karaoke track

## Why this notebook replaces the old one

The ground truth currently bundled in the app was extracted from the
original studio recordings. The app plays **karaoke minus-one** tracks,
which are differently edited. Measured on the ten bundled songs:

| song | karaoke audio | ground truth ends | |
|---|---|---|---|
| Ere - Juan Karlos | 275.8s | 322.3s | **46.5s past the end** |
| Binibini - Zack Tabudlo | 213.6s | 246.7s | **33.1s past the end** |
| Araw-Araw - Ben&Ben | 285.3s | 309.7s | **24.4s past the end** |
| Dadalhin - Alcasid | 283.0s | 294.1s | 11.1s past the end |
| Mula Sa Puso - Regine | 289.9s | 239.0s | stops 50.9s early |

Six of ten describe audio that does not exist in the file being played,
and the start offsets against the hand-synced lyrics scatter from
**-13.33s to +3.55s**. That is not an offset that can be nudged out; the
two recordings are different lengths with different edits.

The **notes were fine**: 20-36 semitone ranges, 4 octave outliers out of
5,317 syllables, median syllable 0.18-0.28s. So this notebook keeps the
same pitch extraction and fixes the clock.

## How

1. Separate vocals from the **original** (the karaoke track has none).
2. Extract pitch (torchcrepe) and syllable timings (Whisper) from that.
3. **Align original -> karaoke with DTW over chroma**, and warp every
   timestamp onto the karaoke timeline.
4. Validate, then export.

Step 3 is the new part. A plain cross-correlation of onsets was tried
before and failed its own control (it estimated -40s for a song whose
true offset is ~0), because correlation can only find a *shift* and these
recordings differ in *length*. DTW over chroma handles both. The warping
code here was tested against synthetic pairs with known edits -- added
intro, trimmed tail, and both at once -- and recovers the shift to within
0.01s.

## What you need

For each song, **two files**:
- the **original** studio recording (has vocals)
- the **karaoke** minus-one the app ships in `assets/audio/`

Set **Runtime -> Change runtime type -> T4 GPU** before you start.

## Cell 1 - install

In [ ]:
# Each install is its own line: when one fails, the rest still run and
# you can see exactly which one broke.
!pip install -q demucs
!pip install -q torchcrepe
!pip install -q openai-whisper
!pip install -q librosa soundfile mutagen

# Deliberately no TensorFlow: torchcrepe is the PyTorch port of CREPE, so
# nothing here needs TF, and installing it alongside Torch on Colab
# fights over the GPU.

import torch
print('torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE -- Runtime > Change runtime type > T4 GPU')

## Cell 2 - upload the pairs

Run it twice: once for originals, once for karaoke versions. You can
select many files in each upload.

**The filenames of the pair must match**, apart from the folder. Easiest
is to name both exactly as the app does: `Ere - Juan Karlos.mp3`.

In [ ]:
import os
import shutil
from google.colab import files

ORIGINAL_DIR = '/content/original'
KARAOKE_DIR = '/content/karaoke'
os.makedirs(ORIGINAL_DIR, exist_ok=True)
os.makedirs(KARAOKE_DIR, exist_ok=True)

WHICH = 'original'   # <<< set to 'original', run, then set to 'karaoke', run again

dest = ORIGINAL_DIR if WHICH == 'original' else KARAOKE_DIR
print(f'Select the {WHICH.upper()} files...')
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join(dest, name))
    print(f'  -> {dest}/{name}')

print(f'\noriginal/: {sorted(os.listdir(ORIGINAL_DIR))}')
print(f'karaoke/ : {sorted(os.listdir(KARAOKE_DIR))}')

## Cell 3 - pair them up and check before spending GPU time

This only reports. Nothing is processed until the pairing looks right.

In [ ]:
import os
import re
from mutagen import File as MutagenFile

ORIGINAL_DIR = '/content/original'
KARAOKE_DIR = '/content/karaoke'
AUDIO_EXT = ('.mp3', '.wav', '.m4a', '.flac', '.ogg')


def stem_of(filename):
    return os.path.splitext(filename)[0].strip()


def duration_of(path):
    info = MutagenFile(path)
    if info is None or not getattr(info, 'info', None):
        return None
    return float(info.info.length)


def listing(folder):
    if not os.path.isdir(folder):
        return {}
    return {stem_of(f): os.path.join(folder, f)
            for f in sorted(os.listdir(folder))
            if f.lower().endswith(AUDIO_EXT)}


originals = listing(ORIGINAL_DIR)
karaokes = listing(KARAOKE_DIR)

PAIRS = {}
print(f"{'song':<40} {'original':>10} {'karaoke':>10} {'diff':>8}")
print('-' * 72)
for stem in sorted(set(originals) | set(karaokes)):
    o, k = originals.get(stem), karaokes.get(stem)
    if o and k:
        do, dk = duration_of(o), duration_of(k)
        PAIRS[stem] = (o, k)
        diff = '' if (do is None or dk is None) else f'{dk - do:+8.1f}'
        print(f'{stem[:40]:<40} {do or 0:10.1f} {dk or 0:10.1f} {diff:>8}')
    elif o:
        print(f'{stem[:40]:<40} {"present":>10} {"MISSING":>10}')
    else:
        print(f'{stem[:40]:<40} {"MISSING":>10} {"present":>10}')

print('-' * 72)
print(f'{len(PAIRS)} complete pair(s) will be processed.')
if not PAIRS:
    print('\nNothing to do. The stems must match exactly on both sides -- '
          'check spelling, dashes and spacing.')
else:
    print('A large "diff" is expected and is the whole reason for this '
          'notebook.')

## Cell 4 - settings and shared helpers

Self-contained: re-run this alone if a later cell complains that a name
is not defined.

In [ ]:
import numpy as np

SR = 16000              # what torchcrepe and Whisper both want
ALIGN_SR = 22050        # chroma alignment; 16k loses little but 22k is standard
HOP_MS = 10             # pitch hop for extraction
ALIGN_HOP = 512         # chroma frame hop, ~23ms at 22050
CONF_MIN = 0.50         # torchcrepe periodicity below this is not a note
MIN_NOTE_MS = 80        # shorter than this is a transient, not a sung note
WHISPER_SIZE = 'small'  # for word timings only; 'medium' if timings look loose
LANG = 'tl'             # per song; set 'en' for the English ones

OUT_DIR = '/content/ground_truth'

NOTE_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']


def hz_to_note(hz):
    """Nearest note name, in the same spelling the app parses."""
    if hz is None or hz <= 0:
        return None
    midi = int(round(69 + 12 * np.log2(hz / 440.0)))
    if not 21 <= midi <= 108:          # outside a piano, so not a sung note
        return None
    return f'{NOTE_NAMES[midi % 12]}{midi // 12 - 1}'


def median_hz(frames):
    """Median in the log domain, so an octave error cannot drag the mean."""
    vals = [f for f in frames if f and f > 0]
    if not vals:
        return None
    return float(2 ** np.median(np.log2(vals)))


print('settings loaded')
print(f'  pitch hop {HOP_MS}ms, confidence floor {CONF_MIN}, '
      f'min note {MIN_NOTE_MS}ms')
print(f'  whisper={WHISPER_SIZE}, lang={LANG}')

## Cell 5 - the alignment code

This is the part that fixes the defect. It was tested against synthetic
pairs with known edits before being put here; the silence floor in
`chroma_of` is there because a silent lead-in gives all-zero chroma and
makes every cosine distance NaN, which aborts the whole alignment.

In [ ]:
import librosa
import numpy as np


def chroma_of(y, sr, hop_length=512):
    """Chroma CQT, L2-normalised per frame.

    Chroma rather than MFCC or onsets: the two recordings share a harmonic
    progression but not a mix -- the karaoke version has no lead vocal,
    often different instrumentation and a different master. Pitch class
    survives all of that.
    """
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop_length)
    # A silent frame has all-zero chroma, and cosine distance to a zero
    # vector is NaN, which makes the DTW cost matrix NaN. Real tracks start
    # and end quiet. A floor turns silence into a uniform vector, which is
    # equidistant from everything, so DTW passes through it.
    chroma = chroma + 1e-6
    norms = np.linalg.norm(chroma, axis=0, keepdims=True)
    return chroma / np.maximum(norms, 1e-8)


def warp_path(y_src, y_dst, sr, hop_length=512):
    """DTW path between two recordings, as (src_sec, dst_sec), ascending."""
    cs = chroma_of(y_src, sr, hop_length)
    cd = chroma_of(y_dst, sr, hop_length)
    _, path = librosa.sequence.dtw(X=cs, Y=cd, metric='cosine')
    path = path[::-1]                     # librosa returns it reversed
    frame_sec = hop_length / sr
    return path[:, 0] * frame_sec, path[:, 1] * frame_sec


def make_mapper(src_sec, dst_sec, dst_duration=None):
    """f(t_original) -> t_karaoke.

    DTW emits several destination frames for one source frame wherever the
    target is slower. Averaging the duplicates gives one monotonic value
    per source time, which keeps the mapping single-valued so a syllable's
    end cannot land before its start.
    """
    src = np.asarray(src_sec, dtype=float)
    dst = np.asarray(dst_sec, dtype=float)
    uniq, inverse = np.unique(src, return_inverse=True)
    collapsed = np.bincount(inverse, weights=dst) / np.bincount(inverse)
    collapsed = np.maximum.accumulate(collapsed)   # never go backwards

    def mapper(t):
        out = np.interp(np.asarray(t, dtype=float), uniq, collapsed)
        if dst_duration is not None:
            out = np.clip(out, 0.0, dst_duration)
        return out

    return mapper


def warp_ground_truth(rows, mapper, dst_duration, min_dur=0.04):
    """Rewrites start/end onto the karaoke timeline; notes untouched.

    Rows landing past the end of the karaoke audio are dropped and
    counted -- that overhang is exactly the defect being fixed, so it gets
    reported rather than silently trimmed.
    """
    starts = mapper([r['start'] for r in rows])
    ends = mapper([r['end'] for r in rows])
    out, dropped = [], 0
    for row, s, e in zip(rows, starts, ends):
        s, e = float(s), float(e)
        if s >= dst_duration:
            dropped += 1
            continue
        e = min(e, dst_duration)
        if e - s < min_dur:
            e = min(s + min_dur, dst_duration)
        if e <= s:
            dropped += 1
            continue
        out.append({**row, 'start': round(s, 3), 'end': round(e, 3)})
    return out, dropped


print('alignment helpers loaded')

## Cell 6 - run it

Demucs, then torchcrepe, then Whisper, then the alignment. Expect a few
minutes per song on a T4. Each song is caught separately, so one failure
does not lose the rest.

In [ ]:
import gc
import os
import re
import subprocess
import traceback

import librosa
import numpy as np
import torch
import torchcrepe
import whisper

os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'loading whisper-{WHISPER_SIZE} on {DEVICE}...')
WHISPER = whisper.load_model(WHISPER_SIZE, device=DEVICE)

RESULTS = {}


def isolate_vocals(path):
    """Demucs two-stem split; returns the vocals wav path."""
    subprocess.run(
        ['python', '-m', 'demucs', '--two-stems', 'vocals',
         '-o', '/content/demucs', path],
        check=True)
    stem = os.path.splitext(os.path.basename(path))[0]
    for model_dir in sorted(os.listdir('/content/demucs')):
        candidate = f'/content/demucs/{model_dir}/{stem}/vocals.wav'
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(f'demucs produced no vocals stem for {stem}')


def pitch_track(vocals_path):
    """torchcrepe over the vocal stem -> (times, hz, periodicity)."""
    audio, sr = torchcrepe.load.audio(vocals_path)
    hop = int(sr * HOP_MS / 1000)
    hz, periodicity = torchcrepe.predict(
        audio, sr, hop_length=hop, fmin=50.0, fmax=1100.0,
        model='full', batch_size=512, device=DEVICE,
        return_periodicity=True)
    hz = hz.squeeze().cpu().numpy()
    periodicity = periodicity.squeeze().cpu().numpy()
    times = np.arange(len(hz)) * HOP_MS / 1000.0
    return times, hz, periodicity


def syllable_rows(vocals_path, times, hz, periodicity, lang):
    """Whisper word timings, each split into syllables, with a note each.

    The note is the log-domain median of the confident pitch frames inside
    the syllable's own slice, which is also how the app picks a note from
    its own frames -- so the target and the measurement are decided the
    same way.
    """
    tr = WHISPER.transcribe(vocals_path, language=lang, word_timestamps=True,
                            condition_on_previous_text=False)
    rows = []
    for seg in tr.get('segments', []):
        for w in seg.get('words', []):
            text = w.get('word', '').strip()
            if not text:
                continue
            w_start, w_end = float(w['start']), float(w['end'])
            if w_end <= w_start:
                continue
            # Vowel groups approximate syllables well enough in Tagalog.
            groups = len([g for g in re.findall(r'[aeiouAEIOU]+', text)]) or 1
            span = (w_end - w_start) / groups
            for i in range(groups):
                s = w_start + i * span
                e = s + span
                mask = (times >= s) & (times < e) & (periodicity >= CONF_MIN)
                note = hz_to_note(median_hz(hz[mask]))
                if note is None:
                    continue
                if (e - s) * 1000 < MIN_NOTE_MS:
                    continue
                rows.append({'syllable': text if groups == 1 else f'{text}',
                             'start': round(s, 3), 'end': round(e, 3),
                             'note': note})
    rows.sort(key=lambda r: r['start'])
    return rows


for stem, (orig_path, kar_path) in PAIRS.items():
    print(f'\n{"=" * 70}\n{stem}\n{"=" * 70}')
    try:
        print('  separating vocals...')
        vocals = isolate_vocals(orig_path)

        print('  pitch tracking...')
        times, hz, periodicity = pitch_track(vocals)

        print('  word timings...')
        rows = syllable_rows(vocals, times, hz, periodicity, LANG)
        print(f'    {len(rows)} syllables on the ORIGINAL timeline')
        if not rows:
            print('    nothing extracted; skipping')
            continue

        print('  aligning original -> karaoke (DTW over chroma)...')
        y_o, _ = librosa.load(orig_path, sr=ALIGN_SR, mono=True)
        y_k, _ = librosa.load(kar_path, sr=ALIGN_SR, mono=True)
        kar_dur = len(y_k) / ALIGN_SR
        src, dst = warp_path(y_o, y_k, ALIGN_SR, ALIGN_HOP)
        mapper = make_mapper(src, dst, kar_dur)
        warped, dropped = warp_ground_truth(rows, mapper, kar_dur)

        shift_start = warped[0]['start'] - rows[0]['start'] if warped else 0
        print(f'    karaoke is {kar_dur:.1f}s long')
        print(f'    first syllable moved {shift_start:+.2f}s')
        print(f'    kept {len(warped)}, dropped {dropped} past the end')
        RESULTS[stem] = warped

    except Exception:
        print(f'  FAILED on {stem}:')
        traceback.print_exc()
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f'\n{len(RESULTS)} song(s) realigned.')

## Cell 7 - validate before you trust any of it

These are the same checks that caught the defect in the old ground truth.
If a song fails one, do not ship that file.

In [ ]:
import re

import numpy as np
from mutagen import File as MutagenFile

NOTE_BASE = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}


def note_to_midi(name):
    m = re.fullmatch(r'([A-G])([#b]?)(-?\d+)', name.strip())
    if not m:
        return None
    semi = NOTE_BASE[m.group(1)]
    semi += 1 if m.group(2) == '#' else (-1 if m.group(2) == 'b' else 0)
    return 12 * (int(m.group(3)) + 1) + semi


print(f"{'song':<34} {'n':>5} {'ends':>7} {'audio':>7} {'range':>6} "
      f"{'bad':>4}  verdict")
print('-' * 86)

for stem, rows in sorted(RESULTS.items()):
    _, kar_path = PAIRS[stem]
    kar_dur = float(MutagenFile(kar_path).info.length)
    ends = max(r['end'] for r in rows)
    midis = [m for m in (note_to_midi(r['note']) for r in rows)
             if m is not None]
    spread = (max(midis) - min(midis)) if midis else 0
    outliers = sum(1 for m in midis if m < 40 or m > 84)
    srt = sorted(rows, key=lambda r: r['start'])
    overlap = sum(1 for a, b in zip(srt, srt[1:]) if b['start'] < a['end'])

    problems = []
    if ends > kar_dur + 0.5:
        problems.append(f'runs {ends - kar_dur:.1f}s past the audio')
    if spread > 40:
        problems.append(f'{spread} semitone range (octave errors?)')
    if outliers > len(rows) * 0.01:
        problems.append(f'{outliers} notes outside a singable range')
    if overlap:
        problems.append(f'{overlap} overlapping syllables')
    if len(rows) < 50:
        problems.append('suspiciously few syllables')

    verdict = 'OK' if not problems else '; '.join(problems)
    print(f'{stem[:34]:<34} {len(rows):5d} {ends:7.1f} {kar_dur:7.1f} '
          f'{spread:6d} {outliers:4d}  {verdict}')

print('-' * 86)
print('"ends" must not exceed "audio" -- that was the original defect.')
print('\nThe check this cannot do for you: play the karaoke track and '
      'confirm the\nfirst syllable lands where the singing actually '
      'starts. Cell 8 prints the\nfirst few so you can compare.')

## Cell 8 - preview, export, download

In [ ]:
import json
import os
import shutil
from google.colab import files

os.makedirs(OUT_DIR, exist_ok=True)

for stem, rows in sorted(RESULTS.items()):
    print(f'\n{stem}  ({len(rows)} syllables)')
    for r in rows[:8]:
        print(f"    {r['start']:7.2f} - {r['end']:7.2f}  "
              f"{r['note']:<4}  {r['syllable']}")

    # Exactly the shape ground_truth_service.dart parses: a top-level
    # "data" list of {syllable, start, end, note}, seconds as numbers.
    payload = {'data': [{'syllable': r['syllable'],
                         'start': r['start'],
                         'end': r['end'],
                         'note': r['note']} for r in rows]}
    out_path = os.path.join(OUT_DIR, f'{stem}.json')
    with open(out_path, 'w', encoding='utf-8') as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=1)

zip_base = '/content/ground_truth_realigned'
shutil.make_archive(zip_base, 'zip', OUT_DIR)
print(f'\nWrote {len(RESULTS)} file(s). Downloading the zip...')
files.download(zip_base + '.zip')

print('\nUnzip into assets/ground_truth/, replacing the old files.')
print('Keep the filenames exactly "<Title> - <Artist>.json" -- the app '
      'resolves\nthem by that convention, not by a registry.')